# Predicting viability from image-based profiles using Elastic Net regression

## Imports, pathing, and constants

In [ ]:
import logging
import pathlib
import sys
import time
import warnings

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import sklearn
import umap
from notebook_init_utils import bandicoot_check, init_notebook
from sklearn.exceptions import ConvergenceWarning
from sklearn.linear_model import ElasticNetCV
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import LeaveOneGroupOut, train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

warnings.filterwarnings("ignore", category=ConvergenceWarning)
root_dir, in_notebook = init_notebook()

if in_notebook:
    import tqdm.notebook as tqdm
else:
    import tqdm

In [ ]:
start_time = time.time()

In [ ]:
# set up logging
LOG_DIR = pathlib.Path("../logs")
LOG_DIR.mkdir(
    parents=True, exist_ok=True
)  # FileHandler errors if this dir doesn't exist

year_month_day_hour_minute_log_name = (
    f"{pd.Timestamp.now().strftime('%Y-%m-%d_%H-%M')}_viability_prediction_training.log"
)
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s - %(levelname)s - %(message)s",
    handlers=[
        logging.FileHandler(LOG_DIR / year_month_day_hour_minute_log_name),
        logging.StreamHandler(sys.stdout),
    ],
    force=True,  # re-applies config if this cell is re-run in the same kernel session
)
# make the logging.info silent to stout
logging.getLogger().setLevel(logging.WARNING)
# logging.disable(logging.INFO)
logging.info("Started viability prediction training run")
logging.info(f"Logging to {LOG_DIR / year_month_day_hour_minute_log_name}")

In [ ]:
"""
Elastic Net viability model: 
training + evaluation under three split strategies

  1. "lopo"          - Leave-One-Patient-Out cross-validation
  2. "loto"          - Leave-One-Treatment-Out cross-validation
  3. "random_split"  - a single random 70/30 train/test split (no grouping)

All three reuse the same cleaning / training / metrics / artifact-saving
logic. For "lopo" and "loto", the model is refit once per fold (holding
out all rows for one patient / one treatment as the test set), metrics
are computed per fold, and results are aggregated across folds. For
"random_split", the model is fit once on a random 70% of rows and
evaluated on the held-out 30%, with the same metrics/artifact format so
it can be compared side-by-side with the grouped results.

Predicted viability is always bounded to [0, 100] before it's reported
or scored, since viability is a percentage and the raw ElasticNet
output is unconstrained and can fall outside that range.

Every saved artifact (metrics/predictions/importances/summary) carries
Metadata_split_method, Metadata_shuffle_status, and Metadata_profile_type
columns, so the combined files produced at the end of the driver loop
are self-describing without needing to parse filenames.
"""

# ---------------------------------------------------------------------
# CONFIG - update these to match your data
# ---------------------------------------------------------------------

RETRAIN_EXISTING_MODELS = (
    False  # if False, will load existing models instead of retraining
)

PATIENT_COL = "Metadata_Biology_PatientTumor"  # column identifying each patient
TREATMENT_COL = "Metadata_Experiment_FullTreatment"  # column identifying each treatment
SPLIT_COL = "Metadata_data_split"  # used only for split_method="predefined"

RANDOM_SPLIT_TEST_SIZE = 0.3  # fraction held out for the random_split test set
RANDOM_SPLIT_SEED = 0  # fixed seed so the random split is reproducible

VIABILITY_MIN = 0.0
VIABILITY_MAX = 100.0

MODEL_OUTPUT = pathlib.Path("../trained_models")
RESULTS_OUTPUT = pathlib.Path("../model_results")
MODEL_OUTPUT.mkdir(exist_ok=True)
RESULTS_OUTPUT.mkdir(exist_ok=True)


def clip_predictions(preds):
    """Bound predicted viability to a physically meaningful [0, 100] range."""
    return np.clip(preds, VIABILITY_MIN, VIABILITY_MAX)


# ---------------------------------------------------------------------
# Data cleaning (from your original script)
# ---------------------------------------------------------------------
def clean_features(df, feature_cols):
    """
    Reports inf/NaN diagnostics for feature_cols, replaces inf with NaN,
    imputes NaN values per feature using the column median, and clips
    extreme values.
    Returns the cleaned dataframe while preserving row count.
    """
    X = df[feature_cols].copy()

    logging.info(f"Any inf in X: {np.isinf(X.values).any()}")
    logging.info(f"Any NaN in X: {np.isnan(X.values).any()}")
    logging.info(f"Max abs value in X: {np.nanmax(np.abs(X.values))}")

    # Index into X's own columns, not df.columns — the boolean mask
    # has one entry per feature column, not per column in df.
    inf_mask = np.isinf(X.values).any(axis=0)
    inf_cols = X.columns[inf_mask]
    logging.info(f"Columns with inf values: {list(inf_cols)}")

    # Replace inf with NaN and impute remaining NaNs column-wise.
    X = X.replace([np.inf, -np.inf], np.nan)
    missing_before = int(X.isna().sum().sum())
    if missing_before > 0:
        median_values = X.median(axis=0, numeric_only=True)
        X = X.fillna(median_values)
        # Any feature with all-NaN values will still be NaN after median fill;
        # fall back to 0 so training can proceed deterministically.
        X = X.fillna(0)
    missing_after = int(X.isna().sum().sum())
    logging.info(
        f"Missing feature values before/after imputation: {missing_before}/{missing_after}"
    )

    # Option B: clip extreme values instead of dropping
    X = X.clip(lower=-1e10, upper=1e10)
    df[feature_cols] = X

    return df.reset_index(drop=True)


# ---------------------------------------------------------------------
# Model training (from your original script)
# ---------------------------------------------------------------------
def train_elastic_net(X_train: pd.DataFrame, Y_train: pd.Series) -> "Pipeline":
    elastic_net_model = make_pipeline(
        StandardScaler(),
        ElasticNetCV(
            l1_ratio=[0.1, 0.2, 0.5, 0.7, 0.9, 0.95, 0.99, 1.0],
            alphas=100,  # int -> auto-generates 100 alphas along the path
            cv=5,
            random_state=0,
            max_iter=50000,
            tol=1e-4,
            n_jobs=-1,
        ),
    )
    elastic_net_model.fit(X_train, Y_train.values.ravel())
    return elastic_net_model


# ---------------------------------------------------------------------
# Metrics (from your original script)
# ---------------------------------------------------------------------
def compute_metrics(model, X, Y) -> dict:
    y_pred = clip_predictions(model.predict(X))
    y_true = Y.values.ravel()
    r2 = r2_score(y_true, y_pred)
    mse = mean_squared_error(y_true, y_pred)
    mae = mean_absolute_error(y_true, y_pred)
    mape = np.mean(np.abs((y_true - y_pred) / y_true)) * 100
    rmse = np.sqrt(mse)
    return {"R2": r2, "MSE": mse, "MAE": mae, "MAPE": mape, "RMSE": rmse}


# ---------------------------------------------------------------------
# Strategy 1 & 2: grouped CV (LOPO / LOTO), reusing the same core logic
# ---------------------------------------------------------------------
def run_group_cv(
    viabilities_df: pd.DataFrame,
    feature_cols: list,
    viability_col: str,
    group_col: str,
    split_name: str,
    **kwargs,
) -> pd.DataFrame:
    """
    Runs Leave-One-Group-Out CV (group_col = patient or treatment column),
    refitting the elastic net pipeline each fold, and saving per-fold and
    aggregated artifacts under OUTPUT_DIR.
    """
    list_of_metadatas = []

    shuffle_status = kwargs.get("shuffle_status", "not_shuffled")
    list_of_metadatas.append(shuffle_status)
    profile_type = kwargs.get("profile_type")
    if profile_type is not None:
        list_of_metadatas.append(profile_type)

    image_mode = kwargs.get("image_mode")
    if image_mode is not None:
        list_of_metadatas.append(image_mode)

    retrain = kwargs.get("retrain")

    tag = "__".join(list_of_metadatas) if list_of_metadatas else split_name

    # Clean BEFORE computing groups, so groups/train_idx/test_idx all line
    # up with the same dataframe.
    if viabilities_df.empty:
        raise ValueError(
            f"Input dataframe is empty before cleaning for split '{split_name}' (profile={profile_type}, shuffle={shuffle_status})."
        )
    viabilities_df = clean_features(viabilities_df, feature_cols)
    if viabilities_df.empty:
        raise ValueError(
            f"No rows remain after cleaning for split '{split_name}' (profile={profile_type}, shuffle={shuffle_status})."
        )

    if group_col not in viabilities_df.columns:
        raise KeyError(f"Grouping column '{group_col}' not found in input dataframe.")

    missing_group_mask = viabilities_df[group_col].isna()
    if missing_group_mask.any():
        dropped = int(missing_group_mask.sum())
        logging.warning(
            f"Dropping {dropped} row(s) with missing group labels in '{group_col}'."
        )
        viabilities_df = viabilities_df.loc[~missing_group_mask].reset_index(drop=True)

    if viabilities_df.empty:
        raise ValueError(
            f"No rows available after removing missing group labels for '{group_col}' (split={split_name})."
        )
    groups = viabilities_df[group_col].values
    unique_groups = np.unique(groups)
    if len(unique_groups) < 2:
        raise ValueError(
            f"Need at least 2 unique groups for LeaveOneGroupOut on '{group_col}', found {len(unique_groups)}."
        )

    logo = LeaveOneGroupOut()

    all_metrics = []
    all_predictions = []
    all_importances = []

    n_splits = logo.get_n_splits(groups=groups)
    logging.info(
        f"=== {split_name} grouped CV on '{group_col}' ({n_splits} folds) for {shuffle_status} "
        f"(profile={profile_type}) ==="
    )

    for fold_idx, (train_idx, test_idx) in enumerate(
        logo.split(viabilities_df, groups=groups)
    ):
        held_out = np.unique(groups[test_idx])[0]

        train_df = viabilities_df.iloc[train_idx]
        test_df = viabilities_df.iloc[test_idx]

        X_train, X_test = train_df[feature_cols], test_df[feature_cols]
        Y_train, Y_test = train_df[viability_col], test_df[viability_col]

        model_output_path = (
            MODEL_OUTPUT / f"{split_name}_model_fold{fold_idx}_{held_out}__{tag}.joblib"
        )
        if model_output_path.exists() and not retrain:
            logging.info(
                f"Model already exists at {model_output_path}, loading it instead of retraining."
            )
            model = joblib.load(model_output_path)
        else:
            model = train_elastic_net(X_train, Y_train)
            joblib.dump(model, model_output_path)

        for eval_split, (X_eval, Y_eval) in {
            "train": (X_train, Y_train),
            "test": (X_test, Y_test),
        }.items():
            m = compute_metrics(model, X_eval, Y_eval)
            m.update(
                {
                    "Metadata_fold": fold_idx,
                    "Metadata_held_out_group": held_out,
                    "Metadata_eval_split": eval_split,
                    "Metadata_n_samples": len(X_eval),
                    "Metadata_split_method": split_name,
                    "Metadata_shuffle_status": shuffle_status,
                    "Metadata_profile_type": profile_type,
                }
            )
            all_metrics.append(m)

            logging.info(
                f"  Fold {fold_idx} (held out={held_out}, n={len(X_eval)}, {eval_split}): "
                f"R2={m['R2']:.4f}, RMSE={m['RMSE']:.4f}"
            )

        # Everything below runs once per FOLD (not once per eval_split) -
        # the model/predictions/coefficients don't change between the
        # train-eval and test-eval passes above.

        # Held-out predictions (bounded to [0, 100] since viability is a percentage)
        fold_preds = test_df.copy()
        fold_preds["Actual_Viability"] = Y_test.values
        fold_preds["Predicted_Viability"] = clip_predictions(model.predict(X_test))
        fold_preds["Metadata_fold"] = fold_idx
        fold_preds["Metadata_held_out_group"] = held_out
        fold_preds["Metadata_split_method"] = split_name
        fold_preds["Metadata_shuffle_status"] = shuffle_status
        fold_preds["Metadata_profile_type"] = profile_type
        all_predictions.append(fold_preds)

        # Feature importances for this fold
        fold_importance = pd.DataFrame(
            {
                "feature": feature_cols,
                "importance": model.named_steps["elasticnetcv"].coef_,
                "Metadata_fold": fold_idx,
                "Metadata_held_out_group": held_out,
                "Metadata_split_method": split_name,
                "Metadata_shuffle_status": shuffle_status,
                "Metadata_profile_type": profile_type,
            }
        )
        all_importances.append(fold_importance)

    metrics_df = pd.DataFrame(all_metrics)[
        [
            "Metadata_fold",
            "Metadata_held_out_group",
            "Metadata_eval_split",
            "Metadata_n_samples",
            "Metadata_split_method",
            "Metadata_shuffle_status",
            "Metadata_profile_type",
            "R2",
            "MSE",
            "MAE",
            "MAPE",
            "RMSE",
        ]
    ]
    metrics_df.to_parquet(
        RESULTS_OUTPUT / f"{split_name}_model_performance__{tag}.parquet", index=False
    )

    predictions_df = pd.concat(all_predictions, ignore_index=True)
    predictions_df.to_parquet(
        RESULTS_OUTPUT / f"{split_name}_predicted_viabilities__{tag}.parquet",
        index=False,
    )

    importances_df = pd.concat(all_importances, ignore_index=True)
    importances_df.to_parquet(
        RESULTS_OUTPUT / f"{split_name}_feature_importances__{tag}.parquet", index=False
    )

    # Aggregate summary (mean/std across folds, test set only)
    test_metrics = metrics_df[metrics_df["Metadata_eval_split"] == "test"]
    summary = test_metrics[["R2", "MSE", "MAE", "MAPE", "RMSE"]].agg(["mean", "std"])
    summary["Metadata_split_method"] = split_name
    summary["Metadata_shuffle_status"] = shuffle_status
    summary["Metadata_profile_type"] = profile_type
    logging.info(
        f"--- {split_name} summary (across {n_splits} folds, test set) ---\n{summary}"
    )
    summary.to_parquet(RESULTS_OUTPUT / f"{split_name}_summary_metrics__{tag}.parquet")

    return metrics_df


# ---------------------------------------------------------------------
# Strategy 3: single random 70/30 train/test split (no grouping)
# ---------------------------------------------------------------------
def run_random_split(
    viabilities_df: pd.DataFrame,
    feature_cols: list,
    viability_col: str,
    split_name: str = "random_split",
    test_size: float = RANDOM_SPLIT_TEST_SIZE,
    random_state: int = RANDOM_SPLIT_SEED,
    **kwargs,
) -> pd.DataFrame:
    """
    Trains once on a random (test_size fraction) held-out split rather than
    grouping by patient or treatment. Rows are shuffled and split
    independently of any Metadata_* grouping column, so the same
    patient/treatment can appear in both train and test — this is meant as
    a baseline to compare against the stricter LOPO/LOTO splits, not a
    substitute for them.

    Saves the same artifact shapes (metrics/predictions/importances/model)
    as run_group_cv, using "Metadata_fold" = 0 and a fixed
    "Metadata_held_out_group" label, so results from both strategies can be
    concatenated and compared directly.
    """
    list_of_metadatas = []

    shuffle_status = kwargs.get("shuffle_status", "not_shuffled")
    list_of_metadatas.append(shuffle_status)

    profile_type = kwargs.get("profile_type")
    if profile_type is not None:
        list_of_metadatas.append(profile_type)

    image_mode = kwargs.get("image_mode")
    if image_mode is not None:
        list_of_metadatas.append(image_mode)

    retrain = kwargs.get("retrain")

    tag = "__".join(list_of_metadatas) if list_of_metadatas else split_name

    if viabilities_df.empty:
        raise ValueError(
            f"Input dataframe is empty before cleaning for split '{split_name}' (profile={profile_type}, shuffle={shuffle_status})."
        )

    viabilities_df = clean_features(viabilities_df, feature_cols)
    if viabilities_df.empty:
        raise ValueError(
            f"No rows remain after cleaning for split '{split_name}' (profile={profile_type}, shuffle={shuffle_status})."
        )
    if len(viabilities_df) < 2:
        raise ValueError(
            f"Need at least 2 rows for train/test split, found {len(viabilities_df)} (split={split_name})."
        )

    held_out_label = f"random_{int(round(test_size * 100))}pct_holdout"

    train_df, test_df = train_test_split(
        viabilities_df, test_size=test_size, random_state=random_state
    )

    logging.info(
        f"=== {split_name} ({int(round((1 - test_size) * 100))}/{int(round(test_size * 100))} "
        f"train/test, n_train={len(train_df)}, n_test={len(test_df)}) for {shuffle_status} "
        f"(profile={profile_type}) ==="
    )

    X_train, X_test = train_df[feature_cols], test_df[feature_cols]
    Y_train, Y_test = train_df[viability_col], test_df[viability_col]

    model_output_path = MODEL_OUTPUT / f"{split_name}_model__{tag}.joblib"
    if model_output_path.exists() and not retrain:
        logging.info(
            f"Model already exists at {model_output_path}, loading it instead of retraining."
        )
        model = joblib.load(model_output_path)
    else:
        model = train_elastic_net(X_train, Y_train)
        joblib.dump(model, model_output_path)

    all_metrics = []
    for eval_split, (X_eval, Y_eval) in {
        "train": (X_train, Y_train),
        "test": (X_test, Y_test),
    }.items():
        m = compute_metrics(model, X_eval, Y_eval)
        m.update(
            {
                "Metadata_fold": 0,
                "Metadata_held_out_group": held_out_label,
                "Metadata_eval_split": eval_split,
                "Metadata_n_samples": len(X_eval),
                "Metadata_split_method": split_name,
                "Metadata_shuffle_status": shuffle_status,
                "Metadata_profile_type": profile_type,
            }
        )
        all_metrics.append(m)
        logging.info(
            f"  {eval_split} (n={len(X_eval)}): R2={m['R2']:.4f}, RMSE={m['RMSE']:.4f}"
        )

    # Held-out predictions (bounded to [0, 100] since viability is a percentage)
    fold_preds = test_df.copy()
    fold_preds["Actual_Viability"] = Y_test.values
    fold_preds["Predicted_Viability"] = clip_predictions(model.predict(X_test))
    fold_preds["Metadata_fold"] = 0
    fold_preds["Metadata_held_out_group"] = held_out_label
    fold_preds["Metadata_split_method"] = split_name
    fold_preds["Metadata_shuffle_status"] = shuffle_status
    fold_preds["Metadata_profile_type"] = profile_type

    fold_importance = pd.DataFrame(
        {
            "feature": feature_cols,
            "importance": model.named_steps["elasticnetcv"].coef_,
            "Metadata_fold": 0,
            "Metadata_held_out_group": held_out_label,
            "Metadata_split_method": split_name,
            "Metadata_shuffle_status": shuffle_status,
            "Metadata_profile_type": profile_type,
        }
    )

    metrics_df = pd.DataFrame(all_metrics)[
        [
            "Metadata_fold",
            "Metadata_held_out_group",
            "Metadata_eval_split",
            "Metadata_n_samples",
            "Metadata_split_method",
            "Metadata_shuffle_status",
            "Metadata_profile_type",
            "R2",
            "MSE",
            "MAE",
            "MAPE",
            "RMSE",
        ]
    ]
    metrics_df.to_parquet(
        RESULTS_OUTPUT / f"{split_name}_model_performance__{tag}.parquet", index=False
    )
    fold_preds.to_parquet(
        RESULTS_OUTPUT / f"{split_name}_predicted_viabilities__{tag}.parquet",
        index=False,
    )
    fold_importance.to_parquet(
        RESULTS_OUTPUT / f"{split_name}_feature_importances__{tag}.parquet", index=False
    )

    # Single-split summary (std will be NaN with only one test evaluation - that's expected)
    test_metrics = metrics_df[metrics_df["Metadata_eval_split"] == "test"]
    summary = test_metrics[["R2", "MSE", "MAE", "MAPE", "RMSE"]].agg(["mean", "std"])
    summary["Metadata_split_method"] = split_name
    summary["Metadata_shuffle_status"] = shuffle_status
    summary["Metadata_profile_type"] = profile_type
    logging.info(f"--- {split_name} summary (single split, test set) ---\n{summary}")
    summary.to_parquet(RESULTS_OUTPUT / f"{split_name}_summary_metrics__{tag}.parquet")

    return metrics_df

In [ ]:
patient_ids = pd.read_csv(
    pathlib.Path(f"{root_dir}/data/patient_IDs.txt").resolve(strict=True),
    header=None,
    sep="\t",
    names=["patient_id"],
)["patient_id"].to_list()

viabilities_path = pathlib.Path(f"{root_dir}/data/viabilities/").resolve(strict=True)

## Combine the profiles, viabilities, and platemap information

In [ ]:
platemap_df_list = []
for patient in patient_ids:
    platemap_file_path = pathlib.Path(
        f"{root_dir}/config/platemaps/{patient}_platemap.csv"
    ).resolve(strict=True)
    tmp_df = pd.read_csv(platemap_file_path, index_col=0)
    tmp_df["patient_id"] = patient
    platemap_df_list.append(tmp_df)
platemap_df = pd.concat(platemap_df_list, axis=0)

In [ ]:
viabilities_df_list = []
for patient in patient_ids:
    viabilities_file_path = pathlib.Path(
        f"{root_dir}/data/viabilities/{patient}_Viabilities.csv"
    ).resolve()
    if not viabilities_file_path.exists():
        continue
    viabilities_df = pd.read_csv(viabilities_file_path)
    # change DMSO dose to 1
    viabilities_df.loc[viabilities_df["Drug"] == "DMSO", "Concentration_uM"] = 1
    viabilities_df.loc[viabilities_df["Drug"] == "PD0325901", "Drug"] = "Mirdametinib"
    viabilities_df["patient_id"] = patient

    viabilities_df_list.append(viabilities_df)
viabilities_df = pd.concat(viabilities_df_list, axis=0)

In [ ]:
# merge the viabilities with the platemap
platemap_viability_df = pd.merge(
    platemap_df,
    viabilities_df,
    how="left",
    left_on=["Treatment", "Dose", "patient_id"],
    right_on=["Drug", "Concentration_uM", "patient_id"],
).drop(columns=["WellCol", "WellPosition"])
# check if the NANs are in B wells only
nan_rows = platemap_viability_df[platemap_viability_df.isna().any(axis=1)]
# drop nan rows
platemap_viability_df = platemap_viability_df.dropna().reset_index(drop=True)
# save the combined platemaps
combined_platemaps_path = pathlib.Path(
    f"{root_dir}/data/viabilities/combined_platemaps.parquet"
).resolve()
platemap_viability_df.to_parquet(combined_platemaps_path, index=False)

## Get all of the morphology profiles to work with

In [ ]:
consensus_profiles_3D_path = pathlib.Path(
    f"{root_dir}/data/profiles_3D/all_patients/3.consensus_profiles/"
).resolve(strict=True)
consensus_profiles_2D_path = pathlib.Path(
    f"{root_dir}/data/profiles_2D/all_patients/"
).resolve(strict=True)
consensus_profiles_3D_df_list = [
    x
    for x in consensus_profiles_3D_path.glob("*.parquet")
    if x.is_file() and "consensus" in x.name and x.suffix == ".parquet"
]
consensus_profiles_2D_df_list = [
    x
    for y in consensus_profiles_2D_path.glob("*")
    if y.is_dir()
    for x in y.iterdir()
    if x.is_file() and "consensus" in x.name and x.suffix == ".parquet"
]
censensus_profiles_all = consensus_profiles_3D_df_list + consensus_profiles_2D_df_list

## Train the models:

In [ ]:
# Initialized ONCE, outside the per-profile loop, so results from every
# profile accumulate instead of being overwritten each iteration.
results = {
    "profile_type": [],
    "split_method": [],
    "shuffle_status": [],
    "results": [],
}

# Fixed seed so the "shuffled" permutation control is reproducible run-to-run.
rng = np.random.default_rng(0)

logging.info(f"Found {len(censensus_profiles_all)} consensus profile(s) to process")

for consensus_path in tqdm.tqdm(
    censensus_profiles_all,
    total=len(censensus_profiles_all),
    desc="Processing consensus profiles",
    leave=True,
):
    consensus_profile_name = consensus_path.stem
    logging.info(f"Processing and training model for {consensus_profile_name}...")
    consensus_df = pd.read_parquet(consensus_path)
    if "2D" in str(consensus_path):
        image_mode = "2D"
        # wrangle the metadata column names to match the 3D consensus profile format
        consensus_df = consensus_df.rename(
            columns={
                "Metadata_patient_tumor": "Metadata_Biology_PatientTumor",
                "Metadata_treatment": "Metadata_Experiment_Treatment",
                "Metadata_dose": "Metadata_Experiment_Dose",
            }
        )
    elif "3D" in str(consensus_path):
        image_mode = "3D"
    else:
        raise ValueError(f"Unexpected consensus profile path: {consensus_path}")

    viabilities_df = (
        pd.merge(
            consensus_df,
            platemap_viability_df,
            how="left",
            left_on=[
                "Metadata_Biology_PatientTumor",
                "Metadata_Experiment_Treatment",
                "Metadata_Experiment_Dose",
            ],
            right_on=["patient_id", "Treatment", "Dose"],
        )
        .drop(
            columns=[
                "Unit",
                "patient_id",
                "Drug",
                "Concentration_uM",
                "Treatment",
                "Dose",
            ]
        )
        .rename(
            columns={
                x: f"Metadata_{x}"
                for x in platemap_viability_df.columns
                if "Viability_percentage" not in x
                and x not in ["WellCol", "WellPosition", "Drug", "Concentration_uM"]
            }
        )
    )
    # drop NAN rows to avoid patients that do not have viability data
    viabilities_df = viabilities_df.dropna(subset=["Viability_percentage"]).reset_index(
        drop=True
    )
    # combine the two stratification columns into a single key
    viabilities_df["Metadata_Experiment_FullTreatment"] = (
        viabilities_df["Metadata_Experiment_Treatment"].astype(str)
        + "_"
        + viabilities_df["Metadata_Experiment_Dose"].astype(str)
    )
    viability_col = ["Viability_percentage"]
    metadata_cols = [
        col for col in viabilities_df.columns if col.startswith("Metadata_")
    ]
    feature_cols = [
        col
        for col in viabilities_df.columns
        if col not in metadata_cols and col not in viability_col
    ]
    viabilities_df[feature_cols] = viabilities_df[feature_cols].clip(
        lower=-1e5, upper=1e5
    )

    for shuffle_status in tqdm.tqdm(
        ["not_shuffled", "shuffled"], desc="Processing shuffle statuses", leave=False
    ):
        if shuffle_status == "shuffled":
            # permute the values in every column (fixed seed via rng, defined above)
            viabilities_df[feature_cols] = viabilities_df[feature_cols].apply(
                lambda col: rng.permutation(col.values)
            )
        for split_method in tqdm.tqdm(
            ["lopo", "loto", "random_split"],
            desc="Processing split methods",
            leave=False,
        ):
            logging.info(
                f"Running split_method={split_method}, shuffle_status={shuffle_status}, "
                f"profile={consensus_profile_name}"
            )

            if split_method == "lopo":
                group_col = PATIENT_COL
            elif split_method == "loto":
                group_col = TREATMENT_COL
            else:
                group_col = None  # not used for "random_split"

            if split_method == "random_split":
                fold_result = run_random_split(
                    viabilities_df,
                    feature_cols,
                    viability_col,
                    split_name=split_method,
                    test_size=RANDOM_SPLIT_TEST_SIZE,
                    random_state=RANDOM_SPLIT_SEED,
                    shuffle_status=shuffle_status,
                    profile_type=consensus_profile_name,
                    retrain=RETRAIN_EXISTING_MODELS,
                    image_mode=image_mode,
                )
            else:
                fold_result = run_group_cv(
                    viabilities_df,
                    feature_cols,
                    viability_col,
                    group_col=group_col,
                    split_name=split_method,
                    shuffle_status=shuffle_status,
                    profile_type=consensus_profile_name,
                    retrain=RETRAIN_EXISTING_MODELS,
                    image_mode=image_mode,
                )

            results["results"].append(fold_result)
            results["profile_type"].append(consensus_profile_name)
            results["split_method"].append(split_method)
            results["shuffle_status"].append(shuffle_status)

            logging.info(
                f"Finished split_method={split_method}, shuffle_status={shuffle_status}, "
                f"profile={consensus_profile_name}"
            )

logging.info(f"Finished processing all {len(censensus_profiles_all)} profile(s)")

In [ ]:
# ---------------------------------------------------------------------
# Concatenate every per-run output into a single combined file per metric,
# now that all profiles / split methods / shuffle statuses have finished.
# ---------------------------------------------------------------------
logging.info(
    "All profiles processed. Concatenating per-run outputs into combined files..."
)

METRIC_FILE_PATTERNS = {
    "model_performance": "*_model_performance__*.parquet",
    "predicted_viabilities": "*_predicted_viabilities__*.parquet",
    "feature_importances": "*_feature_importances__*.parquet",
    "summary_metrics": "*_summary_metrics__*.parquet",
}

combined_paths = {}
for metric_name, pattern in METRIC_FILE_PATTERNS.items():
    matching_files = sorted(RESULTS_OUTPUT.glob(pattern))
    # exclude any combined file from a previous run of this cell
    matching_files = [f for f in matching_files if not f.stem.startswith("combined_")]

    if not matching_files:
        logging.warning(
            f"No files found for pattern '{pattern}' - skipping {metric_name}"
        )
        continue

    dfs = []
    for f in matching_files:
        df = pd.read_parquet(f)
        if metric_name == "summary_metrics":
            # mean/std are stored as the index on these files - promote to a
            # column before concatenating with ignore_index=True, or the
            # mean/std label is lost.
            df = df.reset_index().rename(columns={"index": "Metadata_stat"})
        df["Metadata_source_file"] = f.name
        dfs.append(df)

    combined_df = pd.concat(dfs, ignore_index=True)
    combined_path = RESULTS_OUTPUT / f"combined_{metric_name}.parquet"
    combined_df.to_parquet(combined_path, index=False)
    combined_paths[metric_name] = combined_path

    logging.info(
        f"Wrote {len(combined_df)} rows from {len(matching_files)} file(s) to {combined_path}"
    )

logging.info(
    f"Finished concatenating results into: {[str(p) for p in combined_paths.values()]}"
)
combined_paths

In [ ]:
final_time = time.time()
elapsed_time = final_time - start_time
seconds_gate = 60
minutes_gate = 3600
hours_gate = 3600 * 24
if elapsed_time < seconds_gate:
    logging.info(f"Total elapsed time: {elapsed_time:.2f} seconds")
elif elapsed_time < minutes_gate:
    logging.info(f"Total elapsed time: {elapsed_time / 60:.2f} minutes")
elif elapsed_time < hours_gate:
    logging.info(f"Total elapsed time: {elapsed_time / 3600:.2f} hours")
else:
    logging.info(f"Total elapsed time: {elapsed_time / 86400:.2f} days")